In [ ]:
import numpy as np
import scipy
import scipy.fft as fft
import seaborn as sns

sns.set_theme()

import glob

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import mrcfile
import h5py 
import scipy.constants as constants
import matplotlib.colors as colors

from fnmatch import fnmatch
from helper_functions import (interpolated_intercepts, r_factor, radial, write_text)

e = constants.elementary_charge
h = constants.Planck
c = constants.speed_of_light

saveFig = False
extension = 'prot_only' # 'prot_only' or 'prot_wat'

In [ ]:
base_dir = 'r_factors/'
files_base_dir = sorted(glob.glob(base_dir + f'*_{extensions}_*.mrc',recursive = True))
files_base_dir

In [ ]:
mrc_name = files_base_dir

# Opening undamaged "ideal" Fourier intensities
with mrcfile.open('r_factors/gt/gt.mrc', mode='r') as f_ideal:
    intens_ideal = f_ideal.data
intens_ideal = np.array(intens_ideal)

e_photon_eV = 9000
lambda_photon = (h * c) / (e_photon_eV * e)
d_detector = 0.5
s_pixel = 800e-6 # 800e-6 for 4x

dim = intens_ideal.shape[0]
pixel_num = dim - dim//2
theta_pixel = 0.5 * np.arctan((pixel_num*s_pixel)/d_detector)
resolution = lambda_photon/(2.0*np.sin(theta_pixel))
pix_real = 0.5 * resolution
voxel_size = pix_real

pix_emc = 1 / (intens_ideal.shape[0] * voxel_size * 1e9) # in nm^-1
rad_sh = 5

if rad_sh == 1:
    write_text(f'Size of resolution sphere: {rad_sh} voxel(s) or {pix_emc} nm^-1\n')
else:
    pix_emc *= rad_sh
    write_text(f'Size of resolution sphere: {rad_sh} voxel(s) or {pix_emc} nm^-1\n')
    
# Looping over all aligned Fourier intensities
names_r = []
r_factors = []
emc_list = []

for f in range(len(files_base_dir)):
    # Opening damaged "real" Fourier intensities
    with mrcfile.open(files_base_dir[f], mode='r') as f_real:
        intens_real = f_real.data
    intens_real = np.array(intens_real)
    emc_list.append(intens_real)

    rfs = r_factor(vol_ideal=intens_ideal, vol_real=intens_real, sphere_thickness=rad_sh)
    r_factors.append(rfs)
    
    f_names = files_base_dir[f].split(sep='/')[1].split(sep='.mrc')[0][:]
    names_r.append(f_names)

r_factors = np.array(r_factors)
names_r = np.array(names_r)
emc_list = np.array(emc_list)

In [ ]:
max_p = r_factors.shape[1]
fp_resolution_inv = np.arange(0, max_p) * pix_emc # in nm^-1

plt.figure(dpi=120)
clrs = sns.color_palette('viridis', 5)
for f in range(len(files_base_dir)):
    R_fact = r_factors[f]
    
    # Calculating R-factor with 0.2 threshold intersection
    xcr, ycr = interpolated_intercepts(1/fp_resolution_inv,R_fact,np.repeat(0.2, max_p))
    res_r = xcr
    if res_r.size != 0:
        if res_r.size == 1:
            write_text(f'Resolution for [{f}] {names_r[f]}: {res_r} nm\n')
        else:
            write_text(f'Resolution for [{f}] {names_r[f]}: {res_r[-1]} nm\n')
    else:
        write_text(f'Resolution for [{f}] {names_r[f]} in nm: - nm\n')
    
    plt.plot(fp_resolution_inv, R_fact, '-')
    #plt.xlim([0, fp_resolution_inv[-1]])
    plt.ylim([None, 1.0])
    plt.xlabel(r'$|\mathbf{q}| (nm^{-1})$', weight='bold')
    plt.ylabel('R factor',weight='bold')
    plt.legend([*names_r],frameon=False,prop=dict(weight='bold',size=8.),loc=2)
    
    plt.axhline(y=0.2,xmin=0,xmax=8.5,color='k',linestyle='--',linewidth=1.0,label='_nolegend_')
    plt.axvline(x=1/0.6396483204611855,ymin=0,ymax=1,c='k',linestyle='--',linewidth=1.0,alpha=0.1,label='_nolegend_');
    if saveFig:
        plt.savefig(f'r_factor_all.pdf',transparent=False,bbox_inches='tight',dpi=200);

In [ ]:
# 100k and 1M fill between plot for paper
idx_100k, idx_1M = [], []
for ctx, f in enumerate(names_r):
    if '100k' in f:
        idx_100k.append(ctx)
    elif '1M' in f:
        idx_1M.append(ctx)

idx_100k = np.array(idx_100k)[2:]
idx_1M = np.array(idx_1M)

rfs_100k = r_factors[idx_100k]
rfs_1M = r_factors[idx_1M]

rfactor_min_100k = np.min(rfs_100k, axis=0)
rfactor_max_100k = np.max(rfs_100k, axis=0)

rfactor_min_1M = np.min(rfs_1M, axis=0)
rfactor_max_1M = np.max(rfs_1M, axis=0)

#plt.xlim([0, fp_resolution_inv[-1]])
plt.ylim([None, 1.0])
plt.xlabel(r'$|\mathbf{q}| (nm^{-1})$', weight='bold')
plt.ylabel('R factor',weight='bold')

plt.axhline(y=0.2,xmin=0,xmax=8.5,color='k',linestyle='--',linewidth=1.0,label='_nolegend_')
plt.axvline(x=1/0.6396483204611855,ymin=0,ymax=1,c='k',linestyle='--',linewidth=1.0,alpha=0.1,label='_nolegend_');

plt.fill_between(fp_resolution_inv, rfactor_min_100k, rfactor_max_100k, edgecolor="none", facecolor="b");
plt.fill_between(fp_resolution_inv, rfactor_min_1M, rfactor_max_1M, edgecolor="none", facecolor="r");

if saveFig:
    plt.savefig(f'rfactor_all_{extension}_filled.pdf',transparent=False,bbox_inches='tight',dpi=200);

In [ ]:
cm = 'cividis'
max_v = 0.02
sel_slice = 187

type_emc = 'prot_only' # prot_only or prot_wat

# Ground-truth
condor_slice = intens_ideal[sel_slice,:,:]
plt.figure(dpi=110)
im_0 = plt.imshow(condor_slice,vmin=0,vmax=max_v,cmap=cm,interpolation=None)
plt.xticks([])
plt.yticks([])
minv, maxv = im_0.get_clim()
c_bar_0 = plt.colorbar(fraction=0.05)
c_bar_0.set_ticks([minv,maxv]);
if saveFig:
    plt.savefig(f'results_final/condor_gt.pdf', transparent=False, bbox_inches='tight');

# EMC slice - 100k prot-only or prot-wat
emc_slice = emc_list[1][sel_slice,:,:]
plt.figure(dpi=110)
im_1 = plt.imshow(emc_slice,vmin=0,vmax=max_v,cmap=cm,interpolation=None)
plt.xticks([])
plt.yticks([])
plt.title('100k', weight='bold')
minv, maxv = im_1.get_clim()
if saveFig:
    plt.savefig(f'results_final/emc_100k_{type_emc}.pdf', transparent=False, bbox_inches='tight');

# EMC slice - 1M prot-only or prot-wat
emc_slice = emc_list[3][sel_slice,:,:]
plt.figure(dpi=110)
im_2 = plt.imshow(emc_slice,vmin=0,vmax=max_v,cmap=cm,interpolation=None)
plt.xticks([])
plt.yticks([])
plt.title('1M', weight='bold')
minv, maxv = im_2.get_clim()
if saveFig:
    plt.savefig(f'results_final/emc_1M_{type_emc}.pdf', transparent=False, bbox_inches='tight');

In [ ]:
sel_slice = intens_ideal.shape[0] // 2
sel_emc = 2
num_slices = intens_ideal.shape[0]

fig_handle = plt.figure(constrained_layout = True, dpi = 300)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
fig_handle.suptitle(f'Slice {sel_slice}/{num_slices} for Condor, EMC, and absolute difference plots\n EMC output: {names_r[sel_emc]}', 
            weight='bold', fontsize=6, y=0.82)

cm = 'cividis'

emc_slice = emc_list[sel_emc][:,:,sel_slice]
condor_slice = intens_ideal[:,:,sel_slice]
diff_slice = (condor_slice - emc_slice)

center = sel_slice
scaling = 1.5

circ = Circle(xy=(center, center), color='r', radius=5*25, fill=False)
circ_2 = Circle(xy=(center, center), color='r', radius=5*25, fill=False)

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
max_v = 0.001
im_0 = plt.imshow(condor_slice**scaling,vmin=0,vmax=max_v,cmap=cm,interpolation=None)
ax_0.set_xticks([])
ax_0.set_yticks([])
minv, maxv = im_0.get_clim()
ax_0.set_title(f'Condor',weight='bold',fontsize=5)
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05)
c_bar_0.set_ticks([minv,maxv]);
ax_0.add_patch(circ)

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(emc_slice**scaling,vmin=0,vmax=max_v,cmap=cm,interpolation=None) 
ax_1.set_xticks([]) 
ax_1.set_yticks([]) 
minv, maxv = im_1.get_clim() 
ax_1.set_title(f'EMC',weight='bold',fontsize=5)
c_bar_1 = plt.colorbar(im_1, ax=ax_1,fraction=0.05)
c_bar_1.set_ticks([minv,maxv]);
ax_1.add_patch(circ_2)

ax_2 = fig_handle.add_subplot(spec_handle[0,2])
max_v = 0.01
im_2 = plt.imshow(diff_slice,vmin=-max_v,vmax=max_v,cmap='coolwarm',interpolation=None) 
ax_2.set_xticks([])
ax_2.set_yticks([])
minv, maxv = im_2.get_clim() 
ax_2.set_title(f'Condor - EMC', weight='bold',fontsize=7)
c_bar_2 = plt.colorbar(im_2, ax=ax_2,fraction=0.05)
c_bar_2.set_ticks([minv,maxv]);

# Printing some voxel statistics for debugging
neg_vox = (diff_slice < 0.0).sum()
pos_vox = (diff_slice >= 0.0).sum()

valid_vox = neg_vox + pos_vox
invalid_vox = np.isnan(diff_slice).sum()

tot_vox = diff_slice.shape[0] * diff_slice.shape[1]

write_text(f'Number of negative voxels: {neg_vox}\n')
write_text(f'Number of positive voxels: {pos_vox}\n\n')
write_text(f'Number of valid voxels: {valid_vox}\n')
write_text(f'Number of invalid voxels: {invalid_vox}\n')
write_text(f'Number of total voxels: {tot_vox} (valid + invalid = {valid_vox+invalid_vox})\n')
if saveFig:
    plt.savefig(f'4x_ds_100k_mask.pdf',transparent=False,bbox_inches='tight',dpi=200);